# 01 — FDIC Data Dictionary | Google Colab + Google Drive

Notebook này giữ nguyên **47 biến Financials**, danh sách biến Failures / Institutions / History và toàn bộ **derived features** trong bản gốc của bạn. Các thay đổi chỉ liên quan đến môi trường chạy và đường dẫn.

**Chuẩn bị trên Google Drive** (để nguyên quyền *Restricted*):

```text
MyDrive/FDIC_Bank_Failure_Project/
  metadata/
    risview_properties.yaml
    failure_properties.yaml
    institution_properties.yaml
    history_properties.yaml
  notebooks/
    01_FDIC_Data_Dictionary_Colab.ipynb
```

Bạn cũng có thể giữ tên file YAML có hậu tố `(1)`, ví dụ `risview_properties(1).yaml`; notebook sẽ tự tìm thấy. Folder `metadata/` và file Excel nằm trong dự án trên Drive; notebook tạo thư mục nếu chưa có, nhưng **bạn phải tự tải 4 YAML vào metadata**.

**Chạy:** `Runtime → Run all` trong Google Colab, chấp thuận kết nối Google Drive, sau đó kiểm tra file `FDIC_Data_Dictionary_Updated.xlsx` tại thư mục dự án. Chỉ sử dụng `PyYAML` và `openpyxl`, không cần `artifact_tool`.


In [ ]:
%pip -q install PyYAML openpyxl

In [ ]:
from pathlib import Path

# Colab: thực hiện cấp quyền truy cập My Drive trong phiên hiện tại.
# Không cần đổi quyền chia sẻ thư mục trên Google Drive.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')
else:
    # Fallback khi chạy trong Jupyter / VS Code (để kiểm thử lại nếu cần).
    ROOT = Path.cwd().resolve()
    if ROOT.name.lower() in ('notebooks', 'notebook'):
        ROOT = ROOT.parent

METADATA_DIR = ROOT / 'metadata'
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print('Environment:', 'Google Colab' if IN_COLAB else 'Local Python / Jupyter')
print('Project ROOT:', ROOT)
print('YAML folder:', METADATA_DIR)
print('Available YAML files:', [p.name for p in sorted(METADATA_DIR.glob('*.yaml'))])


Mounted at /content/drive
Environment: Google Colab
Project ROOT: /content/drive/MyDrive/FDIC_Bank_Failure_Project
YAML folder: /content/drive/MyDrive/FDIC_Bank_Failure_Project/metadata
Available YAML files: ['failure_properties.yaml', 'history_properties.yaml', 'institution_properties.yaml', 'risview_properties.yaml']


In [ ]:
from pathlib import Path
import yaml
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ROOT và METADATA_DIR được thiết lập ở cell trước.
YAML_NAMES = {
    'Financials': 'risview_properties.yaml',
    'Failures': 'failure_properties.yaml',
    'Institutions': 'institution_properties.yaml',
    'History': 'history_properties.yaml',
}

def resolve_yaml(filename):
    p = METADATA_DIR / filename
    if p.is_file():
        return p
    # Hỗ trợ tên file giữ nguyên sau khi download, ví dụ ...properties(1).yaml
    alias = METADATA_DIR / f'{Path(filename).stem}(1).yaml'
    return alias if alias.is_file() else p

FILES = {name: resolve_yaml(filename) for name, filename in YAML_NAMES.items()}
missing = [f'{name}: {path.name}' for name, path in FILES.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Không tìm thấy các YAML cần thiết trong Google Drive metadata/: '
        + ', '.join(missing) + '\nFolder hiện tại: ' + str(METADATA_DIR)
        + '\nHãy upload 4 file YAML vào thư mục metadata/ rồi chạy lại từ cell này.'
    )
print('Verified FDIC metadata YAML sources:')
for name, path in FILES.items():
    print(' ', name, '→', path.name)

def make_sheet(wb, name, rows, widths):
    ws = wb.create_sheet(name)
    for row in rows:
        ws.append(row)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 29
    for cell in ws[1]:
        cell.fill = PatternFill('solid', fgColor='15324F')
        cell.font = Font(name='Calibri', bold=True, color='FFFFFF', size=11)
        cell.alignment = Alignment(vertical='center', wrap_text=True)
    for col_idx, width in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = width
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical='top', wrap_text=True)
            if cell.row % 2 == 0:
                cell.fill = PatternFill('solid', fgColor='F2F6FA')
        ws.row_dimensions[row[0].row].height = 46 if name != 'README' else 34
    return ws


Verified FDIC metadata YAML sources:
  Financials → risview_properties.yaml
  Failures → failure_properties.yaml
  Institutions → institution_properties.yaml
  History → history_properties.yaml


## 1. Chọn các biến và định nghĩa vai trò

In [ ]:
FIELDS = {
    'Financials': (
        'CERT REPDTE STNAME '
        'ASSET DEP DEPDOM COREDEP DEPINS DEPUNINS BRO FREPP OTHBRF CHBAL SC SCMV EQ INTAN INTANGW '
        'LNLSGR LNLSNET LNRE LNRECONS LNCOMRE LNREMULT LNRERES LNCI LNCON LNATRES '
        'NCLNLS P3ASSET P9ASSET NAASSET NTLNLS ORE '
        'NETINC INTINC EINTEXP NONII NONIX ELNATR '
        'ROA ROE NIMY RBC1AAJ RBC1RWAJ RBCRWAJ CBLRIND'
    ).split(),
    'Failures': 'CERT NAME CITYST FAILDATE FAILYR COST QBFASSET QBFDEP RESTYPE RESTYPE1'.split(),
    'Institutions': 'CERT NAME STNAME BKCLASS ACTIVE ESTYMD ENDEFYMD CHANGEC1 INSDATE'.split(),
    'History': 'TRANSNUM CHANGECODE CHANGECODE_DESC PROCDATE EFFDATE ENDDATE UNINUM ACQ_UNINUM OUT_UNINUM ACQ_CERT OUT_CERT ORG_ROLE_CDE'.split(),
}

FIN_CATEGORIES = {
    'Identifier': 'CERT REPDTE',
    'Geography': 'STNAME',
    'Balance sheet / funding / liquidity': 'ASSET DEP DEPDOM COREDEP DEPINS DEPUNINS BRO FREPP OTHBRF CHBAL SC SCMV EQ INTAN INTANGW',
    'Loan portfolio / allowance': 'LNLSGR LNLSNET LNRE LNRECONS LNCOMRE LNREMULT LNRERES LNCI LNCON LNATRES',
    'Asset quality': 'NCLNLS P3ASSET P9ASSET NAASSET NTLNLS ORE',
    'Earnings / expenses': 'NETINC INTINC EINTEXP NONII NONIX ELNATR',
    'Profitability / capital ratios': 'ROA ROE NIMY RBC1AAJ RBC1RWAJ RBCRWAJ CBLRIND',
}

DERIVED = [
    # 1. Capital Adequacy (C)
    ('EQ_ASSET', 'Capital', 'EQ / ASSET * 100', 'EQ, ASSET', 'Equity-to-assets ratio (available across full 1984-2026 panel)', '%'),
    ('TANGIBLE_EQ_ASSET', 'Capital', '(EQ - INTAN) / (ASSET - INTAN) * 100', 'EQ, INTAN, ASSET', 'Tangible equity to tangible assets ratio (strips out goodwill/intangibles)', '%'),
    ('GOODWILL_EQ', 'Capital', 'INTANGW / EQ * 100', 'INTANGW, EQ', 'Goodwill share of book equity (captures M&A balance-sheet bloat prior to crisis)', '%'),
    ('TEXAS_RATIO', 'Capital / Asset quality', '(NCLNLS + ORE) / (EQ + LNATRES) * 100', 'NCLNLS, ORE, EQ, LNATRES', 'Nonperforming assets to capital + loan loss reserves (>100% signals severe insolvency risk)', '%'),
    ('NET_NPL_EQUITY', 'Capital / Credit risk', '(NCLNLS - LNATRES) / EQ * 100', 'NCLNLS, LNATRES, EQ', 'Uncovered noncurrent loans relative to equity cushion', '%'),
    ('UNREALIZED_SEC_EQ', 'Capital / Market risk', '(SCMV - SC) / EQ * 100', 'SCMV, SC, EQ', 'Unrealized securities gains/losses relative to book equity (captures 2023 SVB interest-rate risk)', '%'),

    # 2. Asset Quality & Loan Portfolio Concentration (A)
    ('NPL_RATIO', 'Asset quality', 'NCLNLS / LNLSGR * 100', 'NCLNLS, LNLSGR', 'Noncurrent loans to gross loans share', '%'),
    ('EARLY_DELINQ_ASSET', 'Asset quality', 'P3ASSET / ASSET * 100', 'P3ASSET, ASSET', '30-89 days past-due assets to total assets (early leading indicator of distress)', '%'),
    ('PD90_ASSET', 'Asset quality', 'P9ASSET / ASSET * 100', 'P9ASSET, ASSET', '90+ days past-due still accruing assets to total assets', '%'),
    ('NONACCRUAL_ASSET', 'Asset quality', 'NAASSET / ASSET * 100', 'NAASSET, ASSET', 'Nonaccrual assets to total assets', '%'),
    ('ORE_ASSET', 'Asset quality', 'ORE / ASSET * 100', 'ORE, ASSET', 'Other real estate owned (foreclosed property) to total assets', '%'),
    ('ALLOWANCE_LOANS', 'Credit risk', 'LNATRES / LNLSGR * 100', 'LNATRES, LNLSGR', 'Loan loss allowance to gross loans', '%'),
    ('ALLOWANCE_COVERAGE', 'Credit risk', 'LNATRES / (NCLNLS + 1) * 100', 'LNATRES, NCLNLS', 'Loan loss allowance coverage of noncurrent loans (smoothed denominator)', '%'),
    ('NET_CHARGEOFF_RATE', 'Credit risk', 'NTLNLS_Q / LNLSGR * 100', 'NTLNLS, LNLSGR', 'De-cumulated quarterly net charge-offs to gross loans', '%'),
    ('RE_LOAN_SHARE', 'Loan concentration', 'LNRE / LNLSGR * 100', 'LNRE, LNLSGR', 'Total real estate loans share of gross loan portfolio', '%'),
    ('CONSTR_LOAN_ASSET', 'Loan concentration', 'LNRECONS / ASSET * 100', 'LNRECONS, ASSET', 'Construction & land development loans to total assets (RECON in 2009 failure literature)', '%'),
    ('COMRE_LOAN_ASSET', 'Loan concentration', 'LNCOMRE / ASSET * 100', 'LNCOMRE, ASSET', 'Commercial real estate loans to total assets (RECOM in 2009 failure literature)', '%'),
    ('MULTIFAM_LOAN_ASSET', 'Loan concentration', 'LNREMULT / ASSET * 100', 'LNREMULT, ASSET', 'Multifamily residential real estate loans to total assets (REMUL)', '%'),
    ('RES14_LOAN_ASSET', 'Loan concentration', 'LNRERES / ASSET * 100', 'LNRERES, ASSET', '1-4 family residential real estate loans to total assets (RER14)', '%'),
    ('CI_LOAN_SHARE', 'Loan concentration', 'LNCI / LNLSGR * 100', 'LNCI, LNLSGR', 'Commercial & industrial (C&I) loans share of gross loan portfolio', '%'),
    ('CON_LOAN_SHARE', 'Loan concentration', 'LNCON / LNLSGR * 100', 'LNCON, LNLSGR', 'Consumer loans share of gross loan portfolio', '%'),

    # 3. Earnings & Efficiency (E)
    ('INTEREST_EXPENSE_RATIO', 'Earnings', 'EINTEXP_Q / INTINC_Q * 100', 'EINTEXP, INTINC', 'Quarterly interest expense burden relative to interest income', '%'),
    ('PROVISION_RATIO', 'Credit risk / Earnings', 'ELNATR_Q / INTINC_Q * 100', 'ELNATR, INTINC', 'Quarterly credit loss provision burden relative to interest income', '%'),
    ('COST_INCOME', 'Efficiency', 'NONIX_Q / (INTINC_Q - EINTEXP_Q + NONII_Q) * 100', 'NONIX, INTINC, EINTEXP, NONII', 'Quarterly operating cost-to-income ratio (efficiency proxy)', '%'),

    # 4. Liquidity & Funding Structure (L)
    ('DEP_ASSET', 'Funding', 'DEP / ASSET * 100', 'DEP, ASSET', 'Total deposit funding intensity', '%'),
    ('CORE_DEP_ASSET', 'Funding', 'COREDEP / ASSET * 100', 'COREDEP, ASSET', 'Core deposits to total assets (stable retail funding cushion)', '%'),
    ('UNINSURED_DEP_SHARE', 'Funding / Run risk', 'DEPUNINS / DEP * 100', 'DEPUNINS, DEP', 'Uninsured deposits share of total deposits (2023 bank-run vulnerability)', '%'),
    ('FOREIGN_DEP_SHARE', 'Funding', '(DEP - DEPDOM) / DEP * 100', 'DEP, DEPDOM', 'Foreign office deposits share of total deposits', '%'),
    ('BRO_DEP', 'Funding', 'BRO / DEP * 100', 'BRO, DEP', 'Reliance on high-cost, rate-sensitive brokered deposits', '%'),
    ('WHOLESALE_FUNDING_ASSET', 'Funding / Liquidity', '(BRO + FREPP + OTHBRF) / ASSET * 100', 'BRO, FREPP, OTHBRF, ASSET', 'Total non-core wholesale funding reliance (brokered + repo/Fed funds + borrowed money)', '%'),
    ('LOAN_DEPOSIT', 'Liquidity', 'LNLSNET / DEP * 100', 'LNLSNET, DEP', 'Net loans to total deposits ratio (illiquidity measure)', '%'),
    ('CASH_ASSET', 'Liquidity', 'CHBAL / ASSET * 100', 'CHBAL, ASSET', 'Cash and balances due from depository institutions to total assets', '%'),
    ('SEC_ASSET', 'Liquidity', 'SC / ASSET * 100', 'SC, ASSET', 'Investment securities buffer to total assets', '%'),

    # 5. Size, Age & Temporal / Regional Dynamics (S & Window Features for Part 3/4)
    ('ASSET_LOG', 'Size', 'ln(ASSET)', 'ASSET', 'Log total assets (LNSIZE; controls for bank scale and inflation drift)', 'log units'),
    ('BANK_AGE_YEARS', 'Institutional context', '(REPDTE - ESTYMD) / 365.25', 'REPDTE, ESTYMD', 'Bank age in years at reporting quarter (captures De Novo bank fragility)', 'years'),
    ('LOAN_GROWTH_4Q', 'Momentum (4Q Lag)', '(LNLSGR_q - LNLSGR_{q-4}) / LNLSGR_{q-4} * 100', 'CERT, REPDTE, LNLSGR', 'Year-over-year gross loan growth (captures aggressive underwriting expansion)', '%'),
    ('EQ_ASSET_CHG_4Q', 'Momentum (4Q Lag)', 'EQ_ASSET_q - EQ_ASSET_{q-4}', 'CERT, REPDTE, EQ, ASSET', '4-quarter change in equity-to-assets ratio (speed of capital erosion)', 'pp'),
    ('NPL_RATIO_CHG_4Q', 'Momentum (4Q Lag)', 'NPL_RATIO_q - NPL_RATIO_{q-4}', 'CERT, REPDTE, NCLNLS, LNLSGR', '4-quarter surge in noncurrent loan ratio', 'pp'),
    ('ROA_VOL_4Q', 'Sensitivity / Volatility', 'rolling_std_4q(ROA) over CERT', 'CERT, REPDTE, ROA', 'Trailing 4-quarter rolling standard deviation of ROA (earnings instability)', '%'),
    ('STATE_MEDIAN_NPL', 'Regional Macro Proxy', 'median(NPL_RATIO) over (REPDTE, STNAME)', 'REPDTE, STNAME, NCLNLS, LNLSGR', 'Point-in-time state-quarter median NPL ratio (captures regional shocks without look-ahead)', '%'),
]

HEADERS = [
    'Variable Code', 'FDIC Official Title', 'FDIC Official Description',
    'FDIC Type', 'FDIC Unit (if stated)', 'Category', 'Research Role',
    'Source YAML', 'Notes / Cautions', 'Schema Status'
]

def category(code):
    for group, fields in FIN_CATEGORIES.items():
        if code in fields.split():
            return group
    return 'Other'

def role(sheet, code):
    if sheet == 'Financials':
        if code in ('CERT', 'REPDTE'):
            return 'Panel key'
        if code == 'STNAME':
            return 'Context / cross-sectional grouping key'
        return 'Candidate predictor'
    if sheet == 'Failures':
        if code == 'CERT':
            return 'Join key'
        if code == 'FAILDATE':
            return 'Target construction (4-quarter-ahead failure window)'
        if code in ('COST', 'QBFASSET'):
            return 'Cost-based threshold estimation (Dev set only) - NOT predictor'
        return 'Outcome/event metadata - NOT predictor'
    if sheet == 'Institutions':
        if code == 'CERT':
            return 'Join key'
        if code in ('BKCLASS', 'ESTYMD', 'INSDATE'):
            return 'Static institutional context (valid point-in-time predictor if <= REPDTE)'
        return 'End-state / exit classification metadata - NOT predictor'
    return 'Event validation / merger vs. failure exit audit - NOT predictor'

def note(sheet, code, meta):
    notes = []
    if sheet == 'Financials' and code in ('NETINC', 'INTINC', 'EINTEXP', 'NONII', 'NONIX', 'ELNATR', 'NTLNLS'):
        notes.append('YTD flow variable in Call Reports: de-cumulate Q2-Q4 into single-quarter flows (Q_t - Q_{t-1}) within same calendar year before computing ratios')
    if sheet == 'Financials' and code in ('RBC1AAJ', 'RBC1RWAJ', 'RBCRWAJ', 'CBLRIND'):
        notes.append('Regulatory capital metric: structural missingness prior to Basel I (1990s) and under CBLR regime (post-2020); add missingness indicator flag')
    if sheet == 'Financials' and code == 'ROE':
        notes.append('Informative missingness (MNAR): FDIC reports NA when retained earnings are negative; create ROE_is_missing flag before imputation')
    if sheet == 'Financials' and code in ('LNATRES', 'ELNATR'):
        notes.append('LNATRES is balance-sheet allowance stock; ELNATR is income-statement provision flow')
    if sheet == 'Failures' and code in ('COST', 'QBFASSET', 'QBFDEP', 'RESTYPE', 'RESTYPE1'):
        notes.append('Post-failure resolution outcome: severe leakage if used as predictor; use COST on Dev set only to calibrate False Negative cost threshold in Part 5')
    if sheet == 'Institutions' and code in ('ACTIVE', 'ENDEFYMD', 'CHANGEC1'):
        notes.append('Snapshot end-state attribute: use only for panel exit auditing (distinguishing mergers from failures in Part 2), never as model feature')
    if sheet == 'History':
        notes.append('Event-level table: multiple events per CERT possible; aggregate or deduplicate by (CERT, EFFDATE) before joining to panel')
    if not meta.get('description'):
        notes.append('Official YAML description blank; verified via FDIC RIS/Call Report glossary')
    return '; '.join(notes)

def get_properties(path):
    with path.open(encoding='utf-8') as f:
        raw = yaml.safe_load(f)
    return raw['properties']['data']['properties']

## 2. Tạo workbook và trang hướng dẫn

In [ ]:
wb = Workbook()
wb.remove(wb.active)
readme_rows = [
    ['FDIC BANK FAILURE — DATA DICTIONARY', ''],
    ['Purpose', 'Curated metadata, data-quality rules, and point-in-time derived features for quarterly FDIC bank-failure EWS (1984Q1–2026Q2)'],
    ['Source', 'Four official FDIC *_properties.yaml files; official titles/descriptions extracted verbatim'],
    ['Financials', '47 selected variables: 3 panel/context keys + 30 starter Call Report fields + 14 extended CAMELS & crisis-specific fields'],
    ['Failures', 'Failure event metadata; use CERT and FAILDATE for 4Q-ahead labels; use Dev-set COST to calibrate alarm threshold in Part 5'],
    ['Institutions', 'Institution metadata; ESTYMD/BKCLASS provide static context; ACTIVE/CHANGEC1/ENDEFYMD used strictly for non-failure exit auditing'],
    ['History', 'Structural event log (mergers, liquidations, charter changes) used to validate panel exits and prevent false failure labels'],
    ['Derived_Variables', '40 domain-engineered features across CAMELS, RE loan concentration, 4Q momentum, rolling volatility, and regional macro'],
    ['Source_Inventory', 'Automated schema audit comparing total available YAML fields against selected pipeline subset'],
    ['YTD Flow Handling', 'Income/charge-off flows (NETINC, INTINC, EINTEXP, NONII, NONIX, ELNATR, NTLNLS) are YTD cumulative; de-cumulate to quarterly before ratio construction'],
    ['Right-Censoring Rule', 'Download covers 1984Q1–2026Q2, but modelling ends at 2025Q2 to guarantee a complete 4-quarter forward failure window'],
    ['Anti-Leakage Gate', 'All rolling/lag features use strictly past quarters [q-k, q] partitioned by CERT; all imputation/scaling/winsorization fitted on Training split only'],
]
make_sheet(wb, 'README', readme_rows, [33, 115])

<Worksheet "README">

## 3. Đọc YAML và đối chiếu mã biến

In [ ]:
inventory = []
for sheet_name, path in FILES.items():
    props = get_properties(path)
    codes = FIELDS[sheet_name]
    missing = [c for c in codes if c not in props]
    if missing:
        raise ValueError(f'{sheet_name} missing codes: {missing}')
    rows = [HEADERS]
    for code in codes:
        m = props[code] or {}
        rows.append([
            code, str(m.get('title') or ''), str(m.get('description') or ''),
            str(m.get('type') or ''), str(m.get('x-number-unit') or ''),
            category(code) if sheet_name == 'Financials' else sheet_name,
            role(sheet_name, code), path.name, note(sheet_name, code, m),
            'Verified in uploaded YAML'
        ])
    make_sheet(wb, sheet_name, rows, [21, 38, 94, 16, 25, 28, 46, 34, 82, 27])
    inventory.append([sheet_name, path.name, len(props), len(codes), len(missing), 'All selected codes found'])


## 4. Các tỷ số tự tính

In [ ]:
derived_headers = [
    'Derived Code', 'Category', 'Proposed Formula (not FDIC)',
    'Raw Dependencies', 'Interpretation', 'Unit', 'Role', 'Data Quality / Timing Rules'
]
derived_rows = [derived_headers]

for code, cat, formula, deps, meaning, unit in DERIVED:
    if code == 'ASSET_LOG':
        rule = 'Require ASSET > 0; point-in-time at quarter q'
    elif code == 'BANK_AGE_YEARS':
        rule = 'Require valid ESTYMD <= REPDTE; truncate negative anomalies to 0; strictly point-in-time'
    elif '4Q' in code:
        rule = 'Partition by CERT ordered by exact calendar quarter REPDTE; require contiguous 4-quarter history [q-4, q] (do not shift blindly across reporting gaps)'
    elif code == 'STATE_MEDIAN_NPL':
        rule = 'Cross-sectional aggregation grouped strictly by (REPDTE, STNAME) at quarter q; zero look-ahead across future quarters'
    elif code in ('PROVISION_RATIO', 'COST_INCOME', 'INTEREST_EXPENSE_RATIO', 'NET_CHARGEOFF_RATE'):
        rule = 'De-cumulate YTD flow items into single-quarter flows (Q1=YTD1, Q2=YTD2-YTD1, Q3=YTD3-YTD2, Q4=YTD4-YTD3) before dividing; require nonzero denominator'
    else:
        rule = 'Require valid nonzero denominator; handle extreme ratios via training-set-only Winsorization (e.g. 0.5%–99.5%); strictly point-in-time at quarter q'

    derived_rows.append([code, cat, formula, deps, meaning, unit, 'Candidate predictor', rule])

make_sheet(wb, 'Derived_Variables', derived_rows, [27, 24, 57, 39, 68, 16, 24, 98])

<Worksheet "Derived_Variables">

## 5. Xuất Excel lên Google Drive

Workbook được lưu dưới tên **`FDIC_Data_Dictionary_Updated.xlsx`** ngay trong `MyDrive/FDIC_Bank_Failure_Project/`. Notebook 02 sẽ đọc workbook này để tải Financials, Failures, Institutions và History. Không cần đổi quyền truy cập thư mục để notebook dùng được sau khi đã mount Drive.

In [ ]:
# Nếu chạy lại cell này nhiều lần, xóa sheet Source_Inventory cũ (nếu có) để tránh bị nhân bản tên sheet
if 'Source_Inventory' in wb.sheetnames:
    del wb['Source_Inventory']

inventory_rows = [
    ['Dataset', 'YAML file', 'Total fields in YAML', 'Selected fields', 'Missing selected codes', 'Check']
] + inventory

make_sheet(wb, 'Source_Inventory', inventory_rows, [23, 42, 27, 22, 30, 34])

# Đặt tên file Excel mới để lưu bản cập nhật 47 biến Financials + 40 biến Derived
out = ROOT / 'FDIC_Data_Dictionary_Updated.xlsx'
wb.save(out)

print('CREATED NEW FILE:', out)
print('-' * 75)
for row in inventory:
    print(f"{row[0]:<15} | YAML: {row[2]:>4} fields | Selected: {row[3]:>2} fields | {row[5]}")
print('-' * 75)
print('Total raw selected fields across 4 tables:', sum(r[3] for r in inventory))
print('Total derived variables                  :', len(DERIVED))
# Verify output workbook exists on Drive and key sheets are present.
assert out.is_file() and out.stat().st_size > 0, f'Workbook not written: {out}'
from openpyxl import load_workbook
with_sheet = load_workbook(out, read_only=True, data_only=True)
required_sheets = {'README', 'Financials', 'Failures', 'Institutions',
                   'History', 'Derived_Variables', 'Source_Inventory'}
assert required_sheets.issubset(with_sheet.sheetnames), (
    f'Workbook is missing sheets: {required_sheets - set(with_sheet.sheetnames)}')
print('SUCCESS — Excel file verified on:', out)
print('Workbook sheets:', with_sheet.sheetnames)
with_sheet.close()


CREATED NEW FILE: /content/drive/MyDrive/FDIC_Bank_Failure_Project/FDIC_Data_Dictionary_Updated.xlsx
---------------------------------------------------------------------------
Financials      | YAML: 2378 fields | Selected: 47 fields | All selected codes found
Failures        | YAML:   21 fields | Selected: 10 fields | All selected codes found
Institutions    | YAML:  152 fields | Selected:  9 fields | All selected codes found
History         | YAML:  180 fields | Selected: 12 fields | All selected codes found
---------------------------------------------------------------------------
Total raw selected fields across 4 tables: 78
Total derived variables                  : 40
SUCCESS — Excel file verified on: /content/drive/MyDrive/FDIC_Bank_Failure_Project/FDIC_Data_Dictionary_Updated.xlsx
Workbook sheets: ['README', 'Financials', 'Failures', 'Institutions', 'History', 'Derived_Variables', 'Source_Inventory']
